# Improved MADE architectures on a Colab GPU

Connect this notebook to a **Colab GPU** kernel (Select Kernel → Colab → pick GPU, not CPU), then run the cells in order.

Two architecture-improvement candidates, both targeting the MADE paper's *best* scaled result (86.64 ± 0.44 nats, two 8,000-unit layers + 32 masks):

1. **`gated-pixelcnn`** — Gated PixelCNN (van den Oord et al., 2016): vertical + horizontal causal stacks remove the masked-convolution blind spot; gated tanh·sigmoid units replace ReLU. Fixed raster order, 3.84M parameters. Literature anchor: plain PixelCNN reaches 81.30 nats on this dataset.
2. **`lmconv-ensemble`** — order-agnostic locally masked convolutions (Jain et al., 2020): MADE's own two ideas (weight masking + averaging over orderings) realized spatially. Trains by cycling 8 dihedral S-curve orders per batch, evaluates the 8-order ensemble. 635K parameters. Literature anchor: 77.58 nats.

Checkpoints land in `outputs/made/` on the runtime (they disappear when the VM is recycled — download them at the end).

In [1]:
!uv pip install -q torch lightning cyclopts torchmetrics

In [2]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. In Cursor: Select Kernel → Colab → choose a GPU runtime, then rerun."
)
print(torch.cuda.get_device_name(0))
print("torch", torch.__version__)

Tesla T4
torch 2.11.0+cu128


In [3]:
from pathlib import Path
import os

REPO = "https://github.com/ml-and-ds-degree/deep-generative-models-of-texts-and-images.git"
BRANCH = "made-attention"  # change if these commits land on another branch
ROOT = Path("/content/deep-generative-models-of-texts-and-images")

if not (ROOT / "src" / "made_reproduction").exists():
    !git clone --branch {BRANCH} --depth 1 "{REPO}" "{ROOT}"

%cd {ROOT}
!git rev-parse --abbrev-ref HEAD && git log -1 --oneline

os.environ["PYTHONPATH"] = str(Path.cwd() / "src")
print("PYTHONPATH", os.environ["PYTHONPATH"])

/content/deep-generative-models-of-texts-and-images
made-attention
1ca1b72 (grafted, HEAD -> made-attention, origin/made-attention) feat(made): add residual attention MADE and a Colab GPU notebook
PYTHONPATH /content/deep-generative-models-of-texts-and-images/src


## Stage A — Gated PixelCNN

Early stopping (patience 30 on validation NLL) picks the checkpoint; `--max-epochs 150` just bounds the session. Roughly 15–25 s/epoch on a T4.

In [ ]:
!PYTHONPATH=src python -m made_reproduction.cli train binarized-mnist \
  --architecture gated-pixelcnn \
  --accelerator gpu \
  --max-epochs 150 \
  --num-workers 2

In [ ]:
from pathlib import Path

ckpt_dir = Path("outputs/made/binarized_mnist_gated-pixelcnn/checkpoints")
ckpts = sorted(ckpt_dir.glob("epoch-*.ckpt"))
assert ckpts, "No epoch checkpoint yet; wait for training to finish."
gated_best = ckpts[-1]
print("evaluating", gated_best)

!PYTHONPATH=src python -m made_reproduction.cli evaluate "{gated_best}" binarized-mnist --accelerator gpu

## Stage B — Order-agnostic LMConv ensemble

Trains by cycling 8 S-curve orders (one per batch, exactly MADE's mask-resampling protocol). Validation and test average all 8 orders, so validation epochs cost ~8 forward passes per batch.

In [ ]:
!PYTHONPATH=src python -m made_reproduction.cli train binarized-mnist \
  --architecture lmconv-ensemble \
  --accelerator gpu \
  --max-epochs 150 \
  --num-workers 2

In [ ]:
from pathlib import Path

ckpt_dir = Path("outputs/made/binarized_mnist_lmconv-ensemble/checkpoints")
ckpts = sorted(ckpt_dir.glob("epoch-*.ckpt"))
assert ckpts, "No epoch checkpoint yet; wait for training to finish."
lmconv_best = ckpts[-1]
print("evaluating", lmconv_best)

# --masks defaults to the checkpoint's stored value (8-order ensemble).
!PYTHONPATH=src python -m made_reproduction.cli evaluate "{lmconv_best}" binarized-mnist --accelerator gpu

# Single-order NLL for the ablation table:
!PYTHONPATH=src python -m made_reproduction.cli evaluate "{lmconv_best}" binarized-mnist --accelerator gpu --masks 1

## Samples, MMD, and downloads

Same protocol as the report: ancestral samples with seed 1234, unbiased RBF-MMD² with bandwidth² = 196 against held-out test vectors.

In [ ]:
for name, ckpt in (("gated-pixelcnn", gated_best), ("lmconv-ensemble", lmconv_best)):
    !PYTHONPATH=src python -m made_reproduction.cli sample "{ckpt}" \
      --output "outputs/made/samples_{name}.npz" --count 1000 --accelerator gpu
    !PYTHONPATH=src python -m made_reproduction.cli mmd "{ckpt}" binarized-mnist \
      --count 2000 --accelerator gpu

In [ ]:
# Zip everything worth keeping, then download via the Files sidebar
# (or files.download if running in the Colab UI).
!zip -qr improved_runs.zip \
  outputs/made/binarized_mnist_gated-pixelcnn \
  outputs/made/binarized_mnist_lmconv-ensemble \
  outputs/made/samples_gated-pixelcnn.npz \
  outputs/made/samples_lmconv-ensemble.npz
!ls -lh improved_runs.zip